# Tutorial PyTorch tiếng Việt — Từ Tensor đến FashionMNIST

Đây là notebook thực hành dành cho người bắt đầu. Chạy từng ô từ trên xuống. Lần đầu tải tập dữ liệu cần kết nối Internet.

## 0. Cài đặt

Trong terminal: `python -m pip install torch torchvision matplotlib`

Trên Colab, PyTorch thường đã được cài đặt. Nếu cần GPU, bật GPU trong Runtime. Hướng dẫn cài theo thiết bị: https://pytorch.org/get-started/locally/

## 1. Tensor và các phép toán

Tensor là cấu trúc dữ liệu nhiều chiều; `shape` cho biết kích thước từng chiều. `@` nhân ma trận; `*` nhân từng phần tử.

In [1]:
import torch

x = torch.tensor([[1., 2.], [3., 4.]])
y = torch.tensor([[5., 6.], [7., 8.]])
print("shape:", x.shape)
print("x * y:\n", x * y)
print("x @ y:\n", x @ y)

shape: torch.Size([2, 2])
x * y:
 tensor([[ 5., 12.],
        [21., 32.]])
x @ y:
 tensor([[19., 22.],
        [43., 50.]])


## 2. Autograd

Nếu $y=3x^2+2x+1$ thì tại $x=2$, đạo hàm là $14$. Gọi `backward()` để PyTorch tính gradient.

In [2]:
x = torch.tensor(2., requires_grad=True)
y = 3 * x**2 + 2 * x + 1
y.backward()
print("y =", y.item())
print("gradient =", x.grad.item())

y = 17.0
gradient = 14.0


## 3. Chuẩn bị dữ liệu FashionMNIST

FashionMNIST có ảnh xám 28×28 thuộc 10 loại trang phục. Chia tập train gốc thành 54.000 mẫu train, 6.000 validation; giữ 10.000 mẫu test để đánh giá cuối cùng.

In [3]:
# -*- coding: utf-8 -*-
"""Tutorial PyTorch: phân loại FashionMNIST bằng mạng MLP.

Cài đặt:
    python -m pip install torch torchvision

Chạy:
    python pytorch_fashionmnist_tutorial.py

Lần chạy đầu tiên sẽ tải dữ liệu FashionMNIST (~30 MB) từ Internet.
"""

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2

BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 1e-3
SEED = 42
MODEL_PATH = "fashion_mlp_best.pth"

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]




In [4]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")



def make_dataloaders():
    transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])

    full_train = datasets.FashionMNIST(
        root="data", train=True, download=True, transform=transform
    )
    test_set = datasets.FashionMNIST(
        root="data", train=False, download=True, transform=transform
    )

    train_set, val_set = random_split(
        full_train, [54_000, 6_000],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, test_loader



device = get_device()
print("Thiết bị:", device)
# Dữ liệu sẽ được tải khi chạy ô tiếp theo

Thiết bị: mps


## 4. Xây dựng mô hình Neural Network

Mạng MLP nhận ảnh 28×28, làm phẳng thành 784 giá trị, đi qua hai tầng ẩn và xuất 10 logits. Không dùng `softmax` trước `CrossEntropyLoss`.

In [5]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, images):
        # Output là logits (điểm số thô), chưa áp dụng softmax.
        return self.network(images)



model = FashionMLP().to(device)
print(model)
print("Output shape:", model(torch.rand(2, 1, 28, 28, device=device)).shape)

FashionMLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)
Output shape: torch.Size([2, 10])


## 5. Huấn luyện và đánh giá

Mỗi batch: xóa gradient cũ → forward → tính loss → backward → optimizer.step(). Khi đánh giá: `model.eval()` và `torch.inference_mode()`.

In [6]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        n = labels.size(0)
        total_samples += n
        total_loss += loss.item() * n
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / total_samples, total_correct / total_samples


@torch.inference_mode()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        n = labels.size(0)
        total_samples += n
        total_loss += loss.item() * n
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / total_samples, total_correct / total_samples




## 6. Chạy bài toán hoàn chỉnh

Lần chạy đầu tải FashionMNIST; quá trình train có thể lâu hơn trên CPU. Mô hình tốt nhất theo validation accuracy sẽ được lưu trong `fashion_mlp_best.pth`.

In [7]:
def main():
    torch.manual_seed(SEED)
    device = get_device()
    print("Thiết bị:", device)
    print("PyTorch:", torch.__version__)

    train_loader, val_loader, test_loader = make_dataloaders()
    first_images, first_labels = next(iter(train_loader))
    print("Shape ảnh:", tuple(first_images.shape))  # (64, 1, 28, 28)
    print("Shape nhãn:", tuple(first_labels.shape))  # (64,)

    model = FashionMLP().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val_acc = -1.0
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, loss_fn, optimizer, device
        )
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train loss: {train_loss:.4f}, train acc: {train_acc:.2%} | "
            f"val loss: {val_loss:.4f}, val acc: {val_acc:.2%}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)
            print("  -> Đã lưu mô hình tốt nhất:", MODEL_PATH)

    # Chỉ đánh giá tập test sau khi chọn mô hình bằng validation.
    model.load_state_dict(
        torch.load(MODEL_PATH, map_location=device, weights_only=True)
    )
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.2%}")

    # Dự đoán một ảnh chưa sử dụng để huấn luyện.
    image, true_label = test_loader.dataset[0]
    model.eval()
    with torch.inference_mode():
        logits = model(image.unsqueeze(0).to(device))
        probabilities = torch.softmax(logits, dim=1)
        pred_index = probabilities.argmax(dim=1).item()
        confidence = probabilities[0, pred_index].item()

    print("Nhãn thật:", CLASS_NAMES[true_label])
    print("Dự đoán:", CLASS_NAMES[pred_index])
    print(f"Softmax score: {confidence:.2%}")



main()

Thiết bị: mps
PyTorch: 2.14.0


100%|██████████| 26.4M/26.4M [00:04<00:00, 6.14MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 225kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.27MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 2.78MB/s]


Shape ảnh: (64, 1, 28, 28)
Shape nhãn: (64,)
Epoch 01/5 | train loss: 0.5373, train acc: 80.81% | val loss: 0.4204, val acc: 85.10%
  -> Đã lưu mô hình tốt nhất: fashion_mlp_best.pth
Epoch 02/5 | train loss: 0.3766, train acc: 86.29% | val loss: 0.4016, val acc: 84.85%
Epoch 03/5 | train loss: 0.3369, train acc: 87.74% | val loss: 0.3731, val acc: 85.77%
  -> Đã lưu mô hình tốt nhất: fashion_mlp_best.pth
Epoch 04/5 | train loss: 0.3146, train acc: 88.44% | val loss: 0.3413, val acc: 87.52%
  -> Đã lưu mô hình tốt nhất: fashion_mlp_best.pth
Epoch 05/5 | train loss: 0.2934, train acc: 89.22% | val loss: 0.3442, val acc: 87.53%
  -> Đã lưu mô hình tốt nhất: fashion_mlp_best.pth
Test loss: 0.3641 | Test accuracy: 86.99%
Nhãn thật: Ankle boot
Dự đoán: Ankle boot
Softmax score: 99.10%


## 7. Gợi ý luyện tập

1. Thử thay `EPOCHS = 5` thành 10.
2. Đổi `LEARNING_RATE` giữa `1e-2`, `1e-3`, `1e-4` và so sánh.
3. Đổi mạng MLP thành CNN với `Conv2d`.
4. Vẽ biểu đồ train/validation loss sau từng epoch.

**Lưu ý:** Không dùng tập test để chọn siêu tham số. Tài liệu chính thức: https://docs.pytorch.org/tutorials/beginner/basics/intro.html